#### Load the pretrain model and generate GPS Trajectories

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../..')

DEVICE = 'cuda:0'

import random
import matplotlib.pyplot as plt
import os
from tqdm.notebook import tqdm
from diffTraj.utils.Traj_UNet import *
from diffTraj.utils.config import args
from diffTraj.utils.utils import *
from torch.utils.data import DataLoader
from diffTraj.validate import plot_trajectories, generate_diffTraj
from privtrajgen.utils.helpers import dict_to_namespace
from diffTraj.validate import load_diffTraj

# Move to containing directory, but only the first time
if os.getcwd().split('/')[-1] == 'process':
    os.chdir('..')
!pwd

In [ ]:
import logging
from privtrajgen.utils import logger
log = logger.configure_root_loger(logging_level=logging.INFO)
DEVICE = 'cuda:0'

In [ ]:
# Create model
config = dict_to_namespace(args)
unet = load_diffTraj(config, True, param_file='./model.pt', device=DEVICE)

In [ ]:
# Validation Parameters
n_steps = config.diffusion.num_diffusion_timesteps
beta = torch.linspace(config.diffusion.beta_start,
                          config.diffusion.beta_end, n_steps).cuda()
alpha = 1. - beta
alpha_bar = torch.cumprod(alpha, dim=0)
lr = 2e-4  # Explore this - might want it lower when training on the full dataset

eta=0.0
timesteps=100
skip = n_steps // timesteps
seq = range(0, n_steps, skip)

# load conditional information for guide trajectory generation
batchsize = 500
head = np.load('heads.npy', allow_pickle=True)
head = torch.from_numpy(head).float()
dataloader = DataLoader(head, batch_size=batchsize, shuffle=True, num_workers=4)

In [ ]:
# Added by me
print("Shape of head:\t", head.shape)
print("Shape of Batch:\t", next(iter(dataloader)).shape)
print("Sample [0]:\t", head[0])
# Print range for each of the 8 attributes of Head
attr_names = ["Departure Time", "Total Distance", "Total Duration", "Total Length", "Avg Distance", "Average Speed", "Start Grid ID", "End Grid ID\t"]
for i in range(8):
    print(f"Attribute {i+1}:\t{attr_names[i]}\t", head[:, i].min()," - ", head[:, i].max())
    # Compute Mean and STD
    # mean = headd[:, i].mean()
    # std = head[:, i].std()
    # print(f"\t\tMean:\t{mean}\tSTD:\t{std}")

In [ ]:
# Print the names of the attributes one column each for copy and paste
# for i in range(8):
#     print(f"{attr_names[i]}")

In [ ]:
# the mean and std of head information, using for rescaling
hmean=[0, 10283.41600429,   961.66920921,   292.30299616,    36.02766493, 10.98568072]
hstd=[1, 8782.599246414231, 379.41939897358264, 107.24874828393955, 28.749924691281066, 8.774629812281198]
mean = np.array([104.07596303,   30.68085491])
std = np.array([2.15106194e-02, 1.89193207e-02])
# the original mean and std of trajectory length, using for rescaling the trajectory length
len_mean = 292.30299616  # Chengdu
len_std = 107.2487482839  # Chengdu

In [ ]:
functional = True

if functional:
    Gen_traj = generate_diffTraj(model=unet, config=config, device=DEVICE, conditional_dataset=head[:64])
    print("Shape of Gen_traj:\t", len(Gen_traj), Gen_traj[0].shape)
    # Transpose
    Gen_traj = [traj.T for traj in Gen_traj]
else:
    """Following the original code"""
    Gen_traj = []
    Gen_head = []
    head = next(iter(dataloader))
    print("Shape of head:\t", head.shape)
    lengths = head[:, 3]
    print("Shape of Lengths:\t", lengths.shape)
    lengths = (lengths * len_std + len_mean).int()
    tes = head[:,:6].numpy()
    print("Shape of tes:\t", tes.shape)
    Gen_head.extend((tes*hstd+hmean))
    head = head.cuda()
    print("Shape of head:\t", head.shape)
    # Start with random noise
    x = torch.randn(batchsize, 2, config.data.traj_length).cuda()
    print("Shape of x:\t", x.shape)  # (Batch_Size, 2, Length)
    ims = []
    batch_size = x.size(0)
    seq_next = [-1] + list(seq[:-1])
    for i, j in tqdm(zip(reversed(seq), reversed(seq_next)), total=len(seq)):
        t = (torch.ones(batch_size) * i).to(x.device)
        next_t = (torch.ones(batch_size) * j).to(x.device)
        with torch.no_grad():
            pred_noise = unet(x, t, head, verbose=False)
            x = p_xt(x, pred_noise, t, next_t, beta, eta)
            if i % 10 == 0:
                ims.append(x.cpu().squeeze(0))
    print(len(ims), ims[0].shape)
    trajs = ims[-1].cpu().numpy()
    print("Shape of trajs:\t", trajs.shape)
    trajs = trajs[:,:2,:]
    print("Shape of trajs:\t", trajs.shape)
    # resample the trajectory length
    for j in range(batchsize):
        new_traj = resample_trajectory(trajs[j].T, lengths[j])
        # new_traj = new_traj * std + mean
        Gen_traj.append(new_traj)
    print("Shape of Gen_traj:\t", len(Gen_traj), Gen_traj[0].shape)


In [ ]:
plot_trajectories(Gen_traj)
# plt.savefig('Chengdu_traj.png')
plt.show()

In [ ]:
# Plot a single trajectory
plot_trajectories([random.choice(Gen_traj)], alpha=0.5)

## New Code

### No Conditional Info

I would like to know how much the conditional information is affecting the generation of the trajectories. So, I will try to generate the trajectories without the conditional information.
Note that this is not entirely fair as the model has been trained with conditional info and I just feed in some random noise.
There are two ways to replace the conditional information (that we consider here):
1. Use a random head information
    - In this case, we have to distinguish between discrete and continuous attributes
2. Use a zero vector as head information

In [ ]:
def generate(use_random_head): 
    
    # Reversion information Mean and Std of lat/lon
    mean = np.array([104.07596303,   30.68085491])
    std = np.array([2.15106194e-02, 1.89193207e-02])
    # We have to do the following, but IMO this is a privacy leak
    len_mean = 292.30299616  # Chengdu
    len_std = 107.2487482839  # Chengdu
    head = np.load('heads.npy', allow_pickle=True)
    lengths = head[:,3]
    # De normalize the length
    lengths = (lengths * len_std + len_mean).astype(int)
    
    if use_random_head:
        # We use fake head information
        # We have 8 attributes
        # First one is 0-286, last two 0-140. Remainder arbitrary float
        head_replacement = np.random.randn(batchsize, 8)
        head_replacement[:, 0] = np.random.randint(0, 286, batchsize)
        head_replacement[:, -2:] = np.random.randint(0, 140, (batchsize, 2))
        head = torch.from_numpy(head_replacement).float().cuda()
    else:
        # Use a zero vector as head
        head = torch.zeros(batchsize, 8).cuda()
    
    # Only use first batch
    head = head[:64]
    
    generate_trajs = generate_diffTraj(unet, config, DEVICE, head)
    
    # Plot all generated trajectories
    if use_random_head:
        title = "Random Head Information"
    else:
        title = "with Zero Head Information"
        
    plot_trajectories(generate_trajs, title=title)

    
generate(True)

In [ ]:
generate(False)

### Conclusion

As we can see, the output using random information looks quite good. 
On the other hand, the output without conditional information is quite bad, which makes sense as always the same trajectory is generated only differing through the stochastic diffusion process.